# AI Systems 101: Building Agents From Scratch

**What this article is:** A ground-up walkthrough of how AI agents actually work, written in plain Python using only the OpenAI Chat Completions API. No frameworks. No magic. Just code you can read, run, and reason about.

**What this article is not:** A guide to LangChain, the OpenAI Agents SDK, or any other framework. Those come later. You'll be ready for them when you finish here.


---

## Section 0 — Mental Models Before Code

Before writing a single line of Python, we need to establish vocabulary. The word "agent" is used to describe everything from a simple chatbot to a fully autonomous system managing business workflows. That ambiguity causes real confusion, so let's fix it first.

### The Spectrum

Think of AI systems as a spectrum, ordered by who controls what happens next:

| System Type | Who controls the flow? | Example |
|---|---|---|
| **LLM App** | Developer writes one prompt, model answers once | Summariser, classifier |
| **Chatbot** | User and application take turns in a loop | Customer support chat |
| **Augmented LLM** | Application gives the model extra capabilities | Model + tools, or model + structured output |
| **AI Workflow** | Developer code controls the sequence of steps | Classify → extract → draft |
| **Agent** | The model has some control over what happens next | Research assistant deciding which tool to call |
| **Agentic System** | Model + tools + state + guardrails + evaluation | Support case resolver operating autonomously |

The distinction that matters most is this one:

> **In a workflow, the developer controls the steps. In an agent, the model does.**

This does not mean agents are uncontrolled. A well-engineered agent still has constraints, validation, stopping conditions, and human approval where needed. But the *decision about what to do next* belongs to the model, not to an `if/else` statement you wrote.

### When Should You Actually Build an Agent?

Not every problem needs one. A deterministic solution is often faster, cheaper, and easier to debug. Agents earn their complexity when the problem has one or more of these characteristics:

- **Complex decision-making** — the task involves nuanced judgement or context-sensitive exceptions that can't be captured in rules
- **Brittle rule systems** — an existing rules engine has grown unwieldy and expensive to maintain
- **Unstructured data** — the task requires interpreting natural language, extracting meaning from documents, or handling open-ended user input

If your problem doesn't fit one of these, a deterministic solution probably suffices. Don't add unpredictability for no gain.

### The Roadmap

Here is exactly what we're going to build, step by step. Each piece is motivated by the limitation of the one before it.

```
ask_once()             →  a single LLM call, completely stateless
chat_once()            →  same call, but we pass conversation history
analyse_request()      →  same call, but we validate structured output
get_ticket()           →  a plain Python function (a "tool")
TOOLS = {...}          →  a dictionary mapping names to functions
describe_tools()       →  text in a prompt that tells the model tools exist
decide_next_step()     →  a structured LLM call that returns a decision
run_agent()            →  a while loop that assembles all of the above
run_agent() + trace    →  the same loop, now observable
```

By the end, you will have built a working agent in roughly 150 lines of plain Python. Then we will look at what an agent framework does with those same 150 lines — and you will recognise every single abstraction.




---

## Setup

We will use only three libraries. Install them once:

```bash
pip install openai pydantic python-dotenv
```

Create a `.env` file in your working directory:

```
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
```

Then set up your client:

In [ ]:
import json
import os
from typing import Any, Literal
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

load_dotenv()

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI()


That's it. Everything else we build ourselves.


---

## Section 1 — The Simplest Possible LLM Call

Let's start with the absolute minimum: send a message, get a response.


In [ ]:
def ask_once(user_prompt: str) -> str:
    """Send one message to the model and return the reply."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",   "content": user_prompt},
        ],
    )
    return response.choices[0].message.content

reply = ask_once("What is an AI agent?")
print(reply)

This works. For a lot of tasks — summarisation, classification, one-shot extraction — this is genuinely all you need.

But notice what happens if you call it twice:

In [ ]:
ask_once("My name is Alice.")
reply = ask_once("What is my name?")
print(reply)
# → "I don't know your name..."

The model has no memory of the first call. Each call to `ask_once()` is completely independent. The model receives exactly the messages you send, produces a response, and forgets everything.

This is not a bug. It is fundamental to how LLM APIs work:

> **LLM APIs are stateless. Your application is responsible for all context.**

This single constraint is the reason for almost everything that follows.

---

## Section 2 — Messages and Roles

Before we solve the memory problem, we need to understand what we're actually sending to the model.

The OpenAI Chat Completions API takes a list of **messages**. Each message has a `role` and `content`. The roles are:

| Role | Purpose |
|---|---|
| `system` | Sets the model's behaviour, persona, tone, and constraints. Sent at the start. |
| `user` | Represents the human's input. |
| `assistant` | Represents the model's previous responses. |
| `tool` | Represents the result of a tool call returned to the model. (More on this later.) |

A conversation is just a Python list of these dictionaries:

```python
messages = [
    {"role": "system",    "content": "You are a concise assistant."},
    {"role": "user",      "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user",      "content": "And Japan?"},
]
```

When you call the API, you send this entire list. The model reads all of it, then produces the next `assistant` message.

This is the key insight:

> **A "conversation" is not something that lives inside the model. It is a list that lives in your application. You reconstruct it and send it on every call.**

The model has no persistent memory between calls. What looks like memory is just your application passing the history back each time. Once you internalise this, the rest of the article follows naturally.

---
 
## Section 3 — Conversation History and State

Now we can solve the memory problem. The fix is straightforward: maintain a list of messages in our application, append every exchange to it, and send the full list on every call.

In [ ]:
def chat_once(user_input: str, history: list[dict]) -> str:
    """Send one message, update history, return the reply."""

    # Add the user's message to history
    history.append({"role": "user", "content": user_input})

    # Send the full history to the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=history,
    )

    # Extract the reply
    reply = response.choices[0].message.content

    # Add the reply to history so the next call sees it
    history.append({"role": "assistant", "content": reply})

    return reply

Now let's use it:

In [ ]:
history = [
    {"role": "system", "content": "You are a helpful assistant."}
]

chat_once("My name is Alice.", history)
reply = chat_once("What is my name?", history)
print(reply)
# → "Your name is Alice."

It works because `history` now contains both previous exchanges, and we send the whole thing on the second call.

Notice that `history` lives *outside* the function. It is application state — a Python list that we own and control. The model never stores it. We pass it in, and we get an updated list back.

This is the simplest form of memory in an AI system:

> **Memory is not magic. It is application state that you maintain and re-send.**

In production, this list might live in a database, a session store, or a dedicated memory service. For now, a Python list is enough to understand the principle.

We can wrap this in a simple interactive loop:

In [ ]:
def chat_loop() -> None:
    history = [{"role": "system", "content": "You are a helpful assistant."}]
    print("Chat started. Type 'quit' to exit.\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            break
        if not user_input:
            continue
        reply = chat_once(user_input, history)
        print(f"Assistant: {reply}\n")

# chat_loop()

We now have a working chatbot. But try asking it to do something computational:

In [ ]:
chat_once("What is 17% of £340?", history)
# → "17% of £340 is £57.80."  ← sometimes right, sometimes not

The model will attempt arithmetic directly. For simple cases it often gets the right answer, but it can hallucinate, and you cannot rely on it for precise calculation. The model is a language system, not a calculator.

This reveals the next limitation:

> **The model can reason about tasks, but it cannot reliably execute them. It needs capabilities your application provides.**


---

## Section 4 — Structured Outputs

Before we give the model capabilities (tools), we need to solve another problem: the model's output is prose, and prose is hard to act on programmatically.

Consider asking the model to analyse a user request:

In [ ]:
reply = ask_once("Analyse this request: 'What is the refund policy?'")
print(reply)
# → "This request appears to be asking about the company's refund policy.
#    The user likely wants to know the terms and conditions for returning..."

That's a perfectly reasonable response for a human reader. But if your application needs to decide *what to do next* — route to a policy lookup tool, flag a missing piece of information, decide whether calculation is needed — you cannot parse that prose reliably.

The solution is to ask the model to return structured data, and then validate it:

In [ ]:
class RequestAnalysis(BaseModel):
    user_goal: str
    request_type: Literal["general_question", "calculation", "policy_lookup", "unclear"]
    needs_calculation: bool
    needs_policy_lookup: bool
    missing_information: list[str]

def analyse_request(user_request: str) -> RequestAnalysis:
    """Ask the model to analyse a request and return a validated object."""

    prompt = f"""
    Analyse this user request: "{user_request}"

    Return ONLY valid JSON with this exact shape:
    {{
      "user_goal": "string",
      "request_type": "general_question | calculation | policy_lookup | unclear",
      "needs_calculation": true or false,
      "needs_policy_lookup": true or false,
      "missing_information": ["string"]
    }}

    No markdown. No explanation. JSON only.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Return only valid JSON. No markdown fences."},
            {"role": "user",   "content": prompt},
        ],
    )

    raw = response.choices[0].message.content

    try:
        data = json.loads(raw)
        return RequestAnalysis.model_validate(data)
    except (json.JSONDecodeError, ValidationError) as e:
        raise ValueError(f"Model returned invalid output:\n{raw}") from e

Now let's use it:

In [ ]:
analysis = analyse_request("What is the refund policy for premium members?")
print(analysis.model_dump())
# {
#   "user_goal": "Find out the refund policy for premium members",
#   "request_type": "policy_lookup",
#   "needs_calculation": False,
#   "needs_policy_lookup": True,
#   "missing_information": []
# }

The model's response is now a proper Python object your application can act on. You can branch on `request_type`, check flags, validate fields — all reliably.

This pattern is extremely common in production AI systems:

```
natural language input → structured object → application decision
```

It is the bridge between the model's language ability and your application's logic.

But we still haven't solved the core problem: the model can now tell us *what needs to happen*, but it still can't actually do it. It can say `needs_calculation: true` but it cannot calculate. It can say `needs_policy_lookup: true` but it cannot look anything up.

For that, we need tools.


---

## Section 5 — Tools Are Just Functions

"Tools" sounds sophisticated. It isn't. A tool is a plain Python function.

Let's define two:

In [ ]:
# Internal support ticket database
TICKETS = {
    "T-101": {"subject": "Login failure",          "customer_id": "C-09", "status": "open"},
    "T-202": {"subject": "Payment not processed",  "customer_id": "C-14", "status": "open"},
    "T-303": {"subject": "Dashboard not loading",  "customer_id": "C-09", "status": "resolved"},
    "T-404": {"subject": "Export feature broken",  "customer_id": "C-21", "status": "open"},
}

# Customer tier database — internal data the model has never seen
CUSTOMERS = {
    "C-09": {"name": "Acme Corp",       "tier": "enterprise"},
    "C-14": {"name": "Bright Ltd",      "tier": "startup"},
    "C-21": {"name": "Nova Solutions",  "tier": "enterprise"},
}

# SLA policy by customer tier
SLA_POLICIES = {
    "enterprise": "4-hour response, 24-hour resolution",
    "startup":    "8-hour response, 72-hour resolution",
    "free":       "best effort, no guaranteed resolution time",
}

def get_ticket(ticket_id: str) -> str:
    """Look up a support ticket by ID. Returns subject, customer ID and status."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"No ticket found with ID: {ticket_id}"
    return f"Subject: {ticket['subject']}, Customer: {ticket['customer_id']}, Status: {ticket['status']}"

def get_sla_policy(customer_id: str) -> str:
    """Look up the SLA policy for a customer based on their tier."""
    customer = CUSTOMERS.get(customer_id.upper())
    if not customer:
        return f"No customer found with ID: {customer_id}"
    tier   = customer["tier"]
    policy = SLA_POLICIES.get(tier, "No policy found")
    return f"{customer['name']} is on the {tier} tier: {policy}"

Before involving the model at all, test the functions directly:

In [ ]:
print(get_ticket("T-202"))
# → Subject: Payment not processed, Customer: C-14, Status: open

print(get_sla_policy("C-14"))
# → Bright Ltd is on the startup tier: 8-hour response, 72-hour resolution

This is an important engineering habit. Tools should be reliable before you expose them to a model.  
A tool that behaves unpredictably will make your agent unpredictable. Test them like any other function.

> **Note: For a production quality solution, you must implement unit testing.**


Two things to notice:

1. These are ordinary Python functions. There is nothing AI-specific about them.
2. The model is not involved. The model does not run these functions — your application does.

This second point deserves emphasis, because it is the most commonly misunderstood aspect of tool-using agents:

> **The model never executes tools. It requests them. Your application decides whether to run them, and if so, returns the result.**

This distinction matters enormously for safety, reliability, and control. The model is the decision-maker. Your application is the executor.

---

## Section 6 — The Tool Registry

We have two tools. Now we need a way for the application to look up and call the right function when the model requests one.

The simplest possible solution is a dictionary:

In [ ]:
TOOLS: dict[str, callable] = {
    "get_ticket":     get_ticket,
    "get_sla_policy": get_sla_policy,
}

That's it. When the model says "call `get_ticket` with input `T-202`", the application does this:

In [ ]:
tool_name  = "get_ticket"
tool_input = "T-202"

tool_fn = TOOLS[tool_name]       # look up the function
result  = tool_fn(tool_input)    # call it
print(result)                    # Subject: Payment not processed, Customer: C-14, Status: open

In production systems, this registry might be a class with registration decorators, type validation, permission checks, and logging. Frameworks wrap it in sophisticated abstractions. But the core idea is always the same: a mapping from name to callable.

We now have tools defined and a way to call them. The remaining problem is that the model has no idea any of this exists.

---

## Section 7 — Describing Tools to the Model

Tools are just Python functions. But the model is a language system — it only knows what you tell it in text. So to make tools available to the model, you describe them in the system prompt.

This is simpler than it sounds. You write a description of each tool — its name, what it does, and what input it expects — as plain text:

In [ ]:
TOOL_DESCRIPTIONS = """
You are a support operations assistant. You help resolve questions about support tickets and SLA policies.

You have access to the following tools:

- get_ticket(ticket_id: str) -> str
  Looks up a support ticket by ID. Returns the subject, customer ID, and status.
  Example: get_ticket("T-101") → "Subject: Login failure, Customer: C-09, Status: open"

- get_sla_policy(customer_id: str) -> str
  Looks up the SLA policy for a customer based on their tier.
  Example: get_sla_policy("C-09") → "Acme Corp is on the enterprise tier: 4-hour response, 24-hour resolution"

To use a tool, return ONLY valid JSON:
{
  "action": "use_tool",
  "tool_name": "get_ticket",
  "tool_input": "T-101",
  "final_answer": null
}

To give a final answer, return ONLY valid JSON:
{
  "action": "answer",
  "tool_name": null,
  "tool_input": null,
  "final_answer": "Your answer here"
}

Rules: one tool at a time, JSON only, no markdown.
"""

Now we define the structured decision object the model will return:

In [ ]:
class AgentDecision(BaseModel):
    action:       Literal["use_tool", "answer"]
    tool_name:    str | None = None
    tool_input:   str | None = None
    final_answer: str | None = None

And the function that asks the model what to do next:


In [ ]:
def decide_next_step(
    user_request: str,
    observations: list[str],
) -> AgentDecision:
    """Ask the model to decide the next step given the request and observations so far."""

    # Build a summary of what we've observed so far
    observation_text = (
        "\n".join(f"- {o}" for o in observations)
        if observations
        else "None yet."
    )

    prompt = f"""
User request: {user_request}

Observations from previous steps:
{observation_text}

Decide the next step. Return only valid JSON.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": TOOL_DESCRIPTIONS},
            {"role": "user",   "content": prompt},
        ],
    )

    raw = response.choices[0].message.content

    try:
        return AgentDecision.model_validate_json(raw)
    except Exception as e:
        raise ValueError(f"Invalid agent decision:\n{raw}") from e

Let's test it in isolation before building the loop:

In [ ]:
# Step 1: no observations yet
decision = decide_next_step("What is the SLA for ticket T-202?", [])
print(decision)
# → action='use_tool' tool_name='get_ticket' tool_input='T-202'

# Step 2: we know the customer ID now
decision = decide_next_step(
    "What is the SLA for ticket T-202?",
    ["get_ticket(T-202) returned: Subject: Payment not processed, Customer: C-14, Status: open"]
)
print(decision)
# → action='use_tool' tool_name='get_sla_policy' tool_input='C-14'

# Step 3: we have everything
decision = decide_next_step(
    "What is the SLA for ticket T-202?",
    [
        "get_ticket(T-202) returned: Subject: Payment not processed, Customer: C-14, Status: open",
        "get_sla_policy(C-14) returned: Bright Ltd is on the startup tier: 8-hour response, 72-hour resolution"
    ]
)
print(decision)
# → action='answer' ... final_answer='Ticket T-202 is raised by Bright Ltd...'

Notice what is happening here: we are manually stepping through what will become the agent loop. The model is reasoning about what it needs, one step at a time, based solely on what we tell it in the prompt.

Each call to `decide_next_step()` is independent and stateless. The "intelligence" of the agent comes from passing the accumulated observations back in on each call. The model doesn't remember the previous steps — we tell it what happened.

---

## Section 8 — The Agent Loop

We have all the pieces. Now we assemble them.

The pattern is called **ReAct** — Reason, Act, Observe, Repeat, Stop. It maps directly to a `while` loop:

In [ ]:
def run_agent(user_request: str, max_steps: int = 5) -> dict:
    """
    Run a ReAct-style agent loop.

    Each iteration:
      1. REASON  - ask the model what to do next
      2. ACT     - if the model wants a tool, run it
      3. OBSERVE - store the tool result
      4. REPEAT  - go back to step 1 with the new observation
      5. STOP    - when the model answers, or max_steps is reached
    """

    observations = []   # what we have learned from tool calls so far

    for step in range(1, max_steps + 1):

        # --- REASON ---
        # Ask the model what to do next, given what we know so far
        decision = decide_next_step(user_request, observations)

        # --- STOP ---
        # If the model has enough information, return the answer
        if decision.action == "answer":
            return {
                "answer":       decision.final_answer,
                "observations": observations,
                "steps_taken":  step,
            }

        # --- ACT ---
        # The model wants a tool. Look it up and run it.
        if decision.action == "use_tool":

            if decision.tool_name not in TOOLS:
                observations.append(f"Error: unknown tool '{decision.tool_name}'")
                continue

            tool_fn = TOOLS[decision.tool_name]
            result  = tool_fn(decision.tool_input)

            # --- OBSERVE ---
            # Convert the result into an observation string and store it
            observation = (
                f"{decision.tool_name}({decision.tool_input}) returned: {result}"
            )
            observations.append(observation)

    # --- GUARDRAIL ---
    # If we reach here, the loop hit the step limit without answering
    return {
        "answer":       "Stopped: reached maximum number of steps.",
        "observations": observations,
        "steps_taken":  max_steps,
    }

Let's run it:


In [ ]:
result = run_agent("What is the SLA for ticket T-202?")
print(result["answer"])
# → "Ticket T-202 is raised by Bright Ltd, a startup tier customer.
#    The applicable SLA is an 8-hour response and 72-hour resolution time."

print(result["observations"])
# → [
#     "get_ticket(T-202) returned: Subject: Payment not processed, Customer: C-14, Status: open",
#     "get_sla_policy(C-14) returned: Bright Ltd is on the startup tier: 8-hour response, 72-hour resolution"
#   ]

print(result["steps_taken"])
# → 3

Read through `run_agent()` carefully. Every line is explicit.   
There is no hidden machinery. The entire "agent" is:

- A `for` loop
- One function that calls the LLM (`decide_next_step`)
- One dictionary lookup (`TOOLS[name]`)
- One list that accumulates observations

That is the core of every agent system, including the sophisticated ones.   
Frameworks wrap this loop in better abstractions, but the loop is always there.

---

## Section 9 — Stopping Conditions

A critical question when building any agent loop is: *what stops it?*

Without a stopping condition, an agent that cannot find an answer could loop indefinitely, burning API credits and time.     
Even with tools that return `"No ticket found"`, a poorly designed loop might keep trying variations forever — calling the tool repeatedly with slightly different inputs, never reaching a satisfying answer.

Stopping conditions are not optional. They are a fundamental part of agent design:

```python
# In run_agent(), the guardrail at the bottom handles the step limit:
return {
    "answer":       "Stopped: reached maximum number of steps.",
    "observations": observations,
    "steps_taken":  max_steps,
}
```

But `max_steps` is only the most basic stopping condition. In production systems you might also stop on:

- **Error thresholds** — stop if the same tool fails three times
- **Cost limits** — stop if token usage exceeds a budget
- **Time limits** — stop if the task runs longer than N seconds
- **Confidence checks** — stop and escalate to a human if the model signals uncertainty
- **High-risk actions** — pause and require human approval before executing irreversible operations

This points to a broader principle:

> **Guardrails are the application's responsibility, not the model's. The model cannot reliably stop itself. Your code must enforce boundaries.**

The OpenAI guide describes this well: think of guardrails as a layered defence — rules-based checks, LLM-based classifiers, human escalation paths. No single guardrail is sufficient. You layer them.

For now, `max_steps` is enough to illustrate the concept. The important thing is that you are thinking about it explicitly, from the very first loop you write.

---

## Section 10 — The Trace


Run the agent on a few different queries and you will quickly want to know: *what exactly did it do, and why?*

The answer is a **trace** — a record of every decision and observation, step by step.

Adding it to `run_agent()` costs two lines:

In [ ]:
def run_agent(user_request: str, max_steps: int = 5) -> dict:

    observations = []
    trace        = []           # ← add this

    for step in range(1, max_steps + 1):

        decision = decide_next_step(user_request, observations)

        # ← add this block after every decision
        trace_entry = {
            "step":     step,
            "action":   decision.action,
            "tool":     decision.tool_name,
            "input":    decision.tool_input,
            "answer":   decision.final_answer,
        }

        if decision.action == "answer":
            trace.append(trace_entry)
            return {
                "answer":       decision.final_answer,
                "observations": observations,
                "trace":        trace,             # ← return it
                "steps_taken":  step,
            }

        if decision.action == "use_tool":
            if decision.tool_name not in TOOLS:
                observations.append(f"Error: unknown tool '{decision.tool_name}'")
                trace_entry["observation"] = "Error: unknown tool"
                trace.append(trace_entry)
                continue

            tool_fn     = TOOLS[decision.tool_name]
            result      = tool_fn(decision.tool_input)
            observation = f"{decision.tool_name}({decision.tool_input}) returned: {result}"

            observations.append(observation)
            trace_entry["observation"] = observation   # ← record what happened
            trace.append(trace_entry)

    return {
        "answer":       "Stopped: reached maximum number of steps.",
        "observations": observations,
        "trace":        trace,
        "steps_taken":  max_steps,
    }

Now inspect a run:

In [ ]:
result = run_agent("What is the SLA for ticket T-101?")
for entry in result["trace"]:
    print(entry)

# {'step': 1, 'action': 'use_tool', 'tool': 'get_ticket',
#  'input': 'T-101',
#  'observation': 'get_ticket(T-101) returned: Subject: Login failure, Customer: C-09, Status: open'}

# {'step': 2, 'action': 'use_tool', 'tool': 'get_sla_policy',
#  'input': 'C-09',
#  'observation': 'get_sla_policy(C-09) returned: Acme Corp is on the enterprise tier: 4-hour response, 24-hour resolution'}

# {'step': 3, 'action': 'answer', ...,
#  'answer': 'Ticket T-101 is raised by Acme Corp, an enterprise customer.
#             The applicable SLA is a 4-hour response and 24-hour resolution time.'}

You can now see exactly what the agent did, in what order, and what each tool returned. When something goes wrong — and it will — this trace is how you diagnose it.

This is the foundation of observability in AI systems. Production tools like Langfuse, LangSmith, and Weights & Biases Weave are all, at their core, doing this: capturing the trace of an agent run and making it inspectable. They add visualisation, search, alerting, and cost tracking on top. But the data they capture is exactly what we just built.

> **You cannot debug what you cannot see. Add tracing from the very first loop you write.**

---

## Section 11 — Putting It All Together

Here is the complete agent — every concept from Sections 1 through 10 — in one place.

In [ ]:
import json
import os
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
MODEL  = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI()

# ---------------------------------------------------------------------------
# Tools (Section 5)
# ---------------------------------------------------------------------------

# Internal support ticket database
TICKETS = {
    "T-101": {"subject": "Login failure",          "customer_id": "C-09", "status": "open"},
    "T-202": {"subject": "Payment not processed",  "customer_id": "C-14", "status": "open"},
    "T-303": {"subject": "Dashboard not loading",  "customer_id": "C-09", "status": "resolved"},
    "T-404": {"subject": "Export feature broken",  "customer_id": "C-21", "status": "open"},
}

# Customer tier database — internal data the model has never seen
CUSTOMERS = {
    "C-09": {"name": "Acme Corp",       "tier": "enterprise"},
    "C-14": {"name": "Bright Ltd",      "tier": "startup"},
    "C-21": {"name": "Nova Solutions",  "tier": "enterprise"},
}

# SLA policy by customer tier
SLA_POLICIES = {
    "enterprise": "4-hour response, 24-hour resolution",
    "startup":    "8-hour response, 72-hour resolution",
    "free":       "best effort, no guaranteed resolution time",
}

def get_ticket(ticket_id: str) -> str:
    """Look up a support ticket by ID. Returns subject, customer ID and status."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"No ticket found with ID: {ticket_id}"
    return f"Subject: {ticket['subject']}, Customer: {ticket['customer_id']}, Status: {ticket['status']}"

def get_sla_policy(customer_id: str) -> str:
    """Look up the SLA policy for a customer based on their tier."""
    customer = CUSTOMERS.get(customer_id.upper())
    if not customer:
        return f"No customer found with ID: {customer_id}"
    tier   = customer["tier"]
    policy = SLA_POLICIES.get(tier, "No policy found")
    return f"{customer['name']} is on the {tier} tier: {policy}"

# ---------------------------------------------------------------------------
# Tool registry (Section 6)
# ---------------------------------------------------------------------------

TOOLS: dict[str, callable] = {
    "get_ticket":     get_ticket,
    "get_sla_policy": get_sla_policy,
}

# ---------------------------------------------------------------------------
# Tool descriptions (Section 7)
# ---------------------------------------------------------------------------

TOOL_DESCRIPTIONS = """
You are a support operations assistant. You help resolve questions about support tickets and SLA policies.

You have access to the following tools:

- get_ticket(ticket_id: str) -> str
  Looks up a support ticket by ID. Returns the subject, customer ID, and status.
  Example: get_ticket("T-101") → "Subject: Login failure, Customer: C-09, Status: open"

- get_sla_policy(customer_id: str) -> str
  Looks up the SLA policy for a customer based on their tier.
  Example: get_sla_policy("C-09") → "Acme Corp is on the enterprise tier: 4-hour response, 24-hour resolution"

To use a tool, return ONLY valid JSON:
{
  "action": "use_tool",
  "tool_name": "get_ticket",
  "tool_input": "T-101",
  "final_answer": null
}

To give a final answer, return ONLY valid JSON:
{
  "action": "answer",
  "tool_name": null,
  "tool_input": null,
  "final_answer": "Your answer here"
}

Rules: one tool at a time, JSON only, no markdown.
"""

# ---------------------------------------------------------------------------
# Decision model (Section 7)
# ---------------------------------------------------------------------------

class AgentDecision(BaseModel):
    action:       Literal["use_tool", "answer"]
    tool_name:    str | None = None
    tool_input:   str | None = None
    final_answer: str | None = None

# ---------------------------------------------------------------------------
# LLM decision call (Section 7)
# ---------------------------------------------------------------------------

def decide_next_step(
    user_request: str,
    observations: list[str],
) -> AgentDecision:
    """Ask the model to decide the next step given the request and observations so far."""

    # Build a summary of what we've observed so far
    observation_text = (
        "\n".join(f"- {o}" for o in observations)
        if observations
        else "None yet."
    )

    prompt = f"""
User request: {user_request}

Observations from previous steps:
{observation_text}

Decide the next step. Return only valid JSON.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": TOOL_DESCRIPTIONS},
            {"role": "user",   "content": prompt},
        ],
    )

    raw = response.choices[0].message.content

    try:
        return AgentDecision.model_validate_json(raw)
    except Exception as e:
        raise ValueError(f"Invalid agent decision:\n{raw}") from e

# ---------------------------------------------------------------------------
# The agent loop (Sections 8, 9, 10)
# ---------------------------------------------------------------------------

def run_agent(user_request: str, max_steps: int = 5) -> dict:

    observations = []
    trace        = []           # ← add this

    for step in range(1, max_steps + 1):

        decision = decide_next_step(user_request, observations)

        # ← add this block after every decision
        trace_entry = {
            "step":     step,
            "action":   decision.action,
            "tool":     decision.tool_name,
            "input":    decision.tool_input,
            "answer":   decision.final_answer,
        }

        if decision.action == "answer":
            trace.append(trace_entry)
            return {
                "answer":       decision.final_answer,
                "observations": observations,
                "trace":        trace,             # ← return it
                "steps_taken":  step,
            }

        if decision.action == "use_tool":
            if decision.tool_name not in TOOLS:
                observations.append(f"Error: unknown tool '{decision.tool_name}'")
                trace_entry["observation"] = "Error: unknown tool"
                trace.append(trace_entry)
                continue

            tool_fn     = TOOLS[decision.tool_name]
            result      = tool_fn(decision.tool_input)
            observation = f"{decision.tool_name}({decision.tool_input}) returned: {result}"

            observations.append(observation)
            trace_entry["observation"] = observation   # ← record what happened
            trace.append(trace_entry)

    return {
        "answer":       "Stopped: reached maximum number of steps.",
        "observations": observations,
        "trace":        trace,
        "steps_taken":  max_steps,
    }


# ---------------------------------------------------------------------------
# Run it
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    queries = [
        "What is the SLA for ticket T-202?",
        "What are the resolution terms for ticket T-404?",
        "Is ticket T-303 still open, and if so what SLA applies?",
        "What is the SLA policy for ticket T-101?",
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        result = run_agent(query)
        print(f"Answer: {result['answer']}")
        print(f"Steps:  {result['steps_taken']}")
        for entry in result["trace"]:
            print(f"  {entry}")



Read through it slowly. Count the concepts:

- A stateless API call, made stateful by passing context (Sections 1–3)
- Structured output validated with Pydantic (Section 4)
- Plain Python functions as tools (Section 5)
- A dictionary as a tool registry (Section 6)
- Tool descriptions as text in a prompt (Section 7)
- A `for` loop with reason → act → observe (Section 8)
- A `max_steps` guardrail (Section 9)
- A trace list (Section 10)

The entire agent is roughly 150 lines. There are no external agent libraries. Nothing is hidden.


---

## Section 12 — What Frameworks Abstract Away

Now, and only now, let's look at what an agent framework does.

Here is the same support operations agent written using the OpenAI Agents SDK:

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def get_ticket(ticket_id: str) -> str:
    """Look up a support ticket by ID. Returns subject, customer ID and status."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"No ticket found with ID: {ticket_id}"
    return f"Subject: {ticket['subject']}, Customer: {ticket['customer_id']}, Status: {ticket['status']}"

@function_tool
def get_sla_policy(customer_id: str) -> str:
    """Look up the SLA policy for a customer based on their tier."""
    customer = CUSTOMERS.get(customer_id.upper())
    if not customer:
        return f"No customer found with ID: {customer_id}"
    tier   = customer["tier"]
    policy = SLA_POLICIES.get(tier, "No policy found")
    return f"{customer['name']} is on the {tier} tier: {policy}"

support_agent = Agent(
    name="Support Operations Agent",
    instructions="Answer questions about support tickets and SLA policies. Use tools to look up ticket and customer information.",
    tools=[get_ticket, get_sla_policy],
)

result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")
print(result.final_output)

That's it. Roughly 30 lines versus the 150 lines.

But look at what is still there, just wrapped:

| What you built | What the framework provides |
|---|---|
| `TOOLS` dictionary | `@function_tool` decorator registers functions automatically |
| `TOOL_DESCRIPTIONS` in the prompt | Framework generates tool descriptions from docstrings and type hints |
| `AgentDecision` model | Framework handles structured output parsing internally |
| `decide_next_step()` | `Runner.run()` calls the model in a loop |
| `run_agent()` for loop | `Runner.run()` is the loop |
| `max_steps` guardrail | Configurable via `max_turns` parameter |
| `trace` list | Built-in tracing, configurable and exportable |

Nothing has disappeared. The concepts are all still there. The framework has just handled the boilerplate, error cases, edge conditions, and logging that you would otherwise write yourself.

This is why it matters to understand the mechanics first:

> **A framework reduces boilerplate. It does not remove the need to understand what you are building. When something goes wrong — and it will — you need to know what is actually happening inside the loop.**

### Where to Go Next

Now that you understand the foundation, you are ready for the frameworks:

- **OpenAI Agents SDK** — clean, code-first, minimal. Good first framework. The Python you just wrote is almost exactly what it does internally.
- **LangGraph** — graph-based orchestration for more complex multi-agent systems. Explicit nodes and edges give you fine-grained control.
- **Anthropic's tool use API** — Claude's native tool calling, with first-class support for multi-step tool use and structured outputs.

In all of these, you will recognise the loop, the registry, the observations, the trace, and the stopping conditions — because you built them yourself first.

The frameworks are not magic. They are your `run_agent()` function, written by someone who has handled more edge cases than you have had time to encounter yet.

---

## Summary: The 12 Concepts


| Concept | The Problem It Solves |
|---|---|
| The stateless API | Understanding why memory doesn't come for free |
| Messages and roles | Understanding what you actually send to the model |
| Conversation history | Giving the model context across turns |
| Structured outputs | Making model responses reliable enough to act on |
| Tools as functions | Giving the model capabilities your application provides |
| The tool registry | Dispatching tool calls from name to function |
| Describing tools | Telling the model what tools exist |
| The agent loop | Assembling reason → act → observe into a system |
| Stopping conditions | Preventing runaway loops, enforcing guardrails |
| The trace | Making the agent's behaviour inspectable and debuggable |
| The complete agent | Seeing all concepts working together |
| What frameworks abstract | Understanding what you gain — and what you must not forget |



*Next up: Lab Exercises — two hours of hands-on tasks to reinforce each concept above.*